# 01 - Ingestion and ChromaDB collections

Run once. After execution, the three collections are persisted in `data/chroma_db/` and ready for `02_experiments` and `app.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
from pathlib import Path
from collections import Counter
from src.ingestion import ingest_all

chunks = ingest_all(Path('../data/raw'))
print(f'total chunks: {len(chunks)}')
print(Counter(c.metadata.get('source_type') for c in chunks))
print('avg length:', sum(len(c.text) for c in chunks) // max(len(chunks), 1))

In [ ]:
# preview first chunks
for c in chunks[:3]:
    print(c.metadata)
    print(c.text[:200])
    print('---')

In [ ]:
from src.embedding import Embedder
from src.vectorstore import VectorStore

MODELS = {
    'collection_minilm': 'sentence-transformers/all-MiniLM-L6-v2',
    'collection_e5_base': 'intfloat/multilingual-e5-base',
    'collection_solon_base': 'OrdalieTech/Solon-embeddings-base-0.1',
}

for coll_name, model_name in MODELS.items():
    print(f'\n=== {coll_name} ({model_name}) ===')
    emb = Embedder(model_name)
    vs = VectorStore('../data/chroma_db', coll_name)
    if vs.count() > 0:
        print(f'already has {vs.count()} chunks, skip')
        continue
    vs.add_chunks(chunks, emb)
    print(f'stored: {vs.count()}')

In [ ]:
# sanity check: retrieval on a French query
emb = Embedder('sentence-transformers/all-MiniLM-L6-v2')
vs = VectorStore('../data/chroma_db', 'collection_minilm')
res = vs.query("duree de conservation d'un CV", emb, k=3)
for r in res:
    print(r.metadata.get('title'), '|', r.text[:120])